<h3> Path setup and imports </h3>

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
print("PYTHONPATH:", PROJECT_ROOT)


PYTHONPATH: /Users/aidos/ML Projects Personal/Comp_BioChem_Project


In [2]:
import json
from pathlib import Path
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

from src.baselines.features import FeatureConfig, build_features_for_chain, load_labels, load_graph
from src.models.gnn import GNNConfig, InterfaceGNN

ROOT = PROJECT_ROOT
META = ROOT / "data/metadata"
PROCESSED = ROOT / "data/processed"
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Device:", device)


Device: mps


<h3> Load split</h3>

In [3]:
with open(META / "data_splits.json", "r") as f:
    splits = json.load(f)

len(splits["train"]), len(splits["val"]), len(splits["test"])


(180, 35, 35)

<h3> Dataset class (returns one chain-graph at a time)

In [4]:
class ChainGraphExample:
    def __init__(self, x, edge_index, edge_dist, y, meta):
        self.x = x
        self.edge_index = edge_index
        self.edge_dist = edge_dist
        self.y = y
        self.meta = meta

class PPICChainDataset(Dataset):
    def __init__(self, split_list, split_name: str, t_angstrom=5):
        self.items = []
        self.cfg = FeatureConfig(include_flags=True, include_position=True)

        for ex in split_list:
            ex_dir = PROCESSED / f'{ex["pdb_id"]}_{ex["chainA"]}_{ex["chainB"]}'
            for chain in ["A", "B"]:
                X, feat_names = build_features_for_chain(ex_dir, chain, cfg=self.cfg)
                y = load_labels(ex_dir, chain, t_angstrom=t_angstrom)

                edge_index, edge_dist = load_graph(ex_dir, chain)

                # convert to torch
                x_t = torch.tensor(X, dtype=torch.float32)
                y_t = torch.tensor(y, dtype=torch.float32)

                ei_t = torch.tensor(edge_index, dtype=torch.long)
                ed_t = torch.tensor(edge_dist, dtype=torch.float32)

                meta = {"example": ex_dir.name, "chain": chain, "split": split_name}
                self.items.append(ChainGraphExample(x_t, ei_t, ed_t, y_t, meta))

        self.feat_names = feat_names

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        return self.items[idx]

def collate_one(batch):
    # We train one graph at a time for simplicity and clarity.
    assert len(batch) == 1
    return batch[0]


<h3> Building Loaders</h3>

In [5]:
train_ds = PPICChainDataset(splits["train"], "train", t_angstrom=5)
val_ds   = PPICChainDataset(splits["val"],   "val",   t_angstrom=5)
test_ds  = PPICChainDataset(splits["test"],  "test",  t_angstrom=5)

train_loader = DataLoader(train_ds, batch_size=1, shuffle=True, collate_fn=collate_one)
val_loader   = DataLoader(val_ds,   batch_size=1, shuffle=False, collate_fn=collate_one)
test_loader  = DataLoader(test_ds,  batch_size=1, shuffle=False, collate_fn=collate_one)

len(train_ds), len(val_ds), len(test_ds), train_ds.feat_names[:5]


(360, 70, 70, ['aa_A', 'aa_R', 'aa_N', 'aa_D', 'aa_C'])

<h3> Compute pos_weight from train split (weighted BCE)</h3>

In [6]:
# pos_weight = (#neg / #pos) computed over all train residues
pos = 0
neg = 0
for item in train_ds.items:
    pos += int(item.y.sum().item())
    neg += int((item.y.numel() - item.y.sum()).item())

pos_weight = torch.tensor([neg / max(1, pos)], dtype=torch.float32, device=device)
pos, neg, pos_weight


(9430, 71504, tensor([7.5826], device='mps:0'))

<h3> Initializing Model + Optimizer

In [7]:
in_dim = train_ds.items[0].x.shape[1]
cfg = GNNConfig(in_dim=in_dim, hidden_dim=64, num_layers=3, num_rbf=12, rbf_dmin=2.0, rbf_dmax=10.0, dropout=0.10)

model = InterfaceGNN(cfg).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)


<h3> Evaluation metrics helpers (PR-AUC + Precision@K)

In [8]:
from sklearn.metrics import average_precision_score

def eval_loader(loader, k_list=(10,20,30)):
    model.eval()
    praucs = []
    p_at = {k: [] for k in k_list}
    f1_at = {k: [] for k in k_list}

    with torch.no_grad():
        for item in loader:
            x = item.x.to(device)
            ei = item.edge_index.to(device)
            ed = item.edge_dist.to(device)
            y = item.y.to(device)

            logits = model(x, ei, ed)
            score = torch.sigmoid(logits).detach().cpu().numpy()
            y_np  = y.detach().cpu().numpy().astype(np.int8)

            pr = average_precision_score(y_np, score)
            praucs.append(pr)

            for k in k_list:
                idx = np.argsort(-score)[:k]
                pred = np.zeros_like(y_np)
                pred[idx] = 1

                tp = ((pred==1) & (y_np==1)).sum()
                fp = ((pred==1) & (y_np==0)).sum()
                fn = ((pred==0) & (y_np==1)).sum()

                prec = tp / max(1, k)
                rec = tp / max(1, (tp+fn))
                f1 = 0.0 if (prec+rec)==0 else (2*prec*rec/(prec+rec))

                p_at[k].append(float(prec))
                f1_at[k].append(float(f1))

    out = {"prauc": float(np.mean(praucs))}
    for k in k_list:
        out[f"p@{k}"] = float(np.mean(p_at[k]))
        out[f"f1@{k}"] = float(np.mean(f1_at[k]))
    return out


<h3> Training Loop (early stopping on val PR-AUC)</h3>

In [9]:
best_val = -1.0
best_state = None

for epoch in range(1, 21):
    model.train()
    losses = []

    for item in tqdm(train_loader, desc=f"epoch {epoch}", leave=False):
        x = item.x.to(device)
        ei = item.edge_index.to(device)
        ed = item.edge_dist.to(device)
        y = item.y.to(device)

        opt.zero_grad()
        logits = model(x, ei, ed)
        loss = criterion(logits, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
        opt.step()

        losses.append(float(loss.item()))

    val_metrics = eval_loader(val_loader)
    train_loss = float(np.mean(losses))

    print(f"Epoch {epoch:02d} | train_loss={train_loss:.4f} | val_prauc={val_metrics['prauc']:.4f} | "
          f"val_p@10={val_metrics['p@10']:.3f} val_p@20={val_metrics['p@20']:.3f} val_p@30={val_metrics['p@30']:.3f}")

    if val_metrics["prauc"] > best_val:
        best_val = val_metrics["prauc"]
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

# restore best
model.load_state_dict(best_state)
print("Best val PR-AUC:", best_val)


epoch 1:   0%|          | 0/360 [00:00<?, ?it/s]

Epoch 01 | train_loss=1.3869 | val_prauc=0.3781 | val_p@10=0.467 val_p@20=0.428 val_p@30=0.416


epoch 2:   0%|          | 0/360 [00:00<?, ?it/s]

Epoch 02 | train_loss=1.3435 | val_prauc=0.3737 | val_p@10=0.434 val_p@20=0.413 val_p@30=0.403


epoch 3:   0%|          | 0/360 [00:00<?, ?it/s]

...

Epoch 18 | train_loss=1.1035 | val_prauc=0.3949 | val_p@10=0.410 val_p@20=0.412 val_p@30=0.398


epoch 19:   0%|          | 0/360 [00:00<?, ?it/s]

Epoch 19 | train_loss=1.0954 | val_prauc=0.4100 | val_p@10=0.470 val_p@20=0.459 val_p@30=0.437


epoch 20:   0%|          | 0/360 [00:00<?, ?it/s]

Epoch 20 | train_loss=1.0707 | val_prauc=0.4144 | val_p@10=0.484 val_p@20=0.464 val_p@30=0.450
Best val PR-AUC: 0.41444980873526505


<h3> Final evaluation on test

In [10]:
val_final = eval_loader(val_loader)
test_final = eval_loader(test_loader)

val_final, test_final


({'prauc': 0.41444980873526505,
  'p@10': 0.48428571428571415,
  'f1@10': 0.12771253300356875,
  'p@20': 0.4642857142857143,
  'f1@20': 0.21497766004141625,
  'p@30': 0.4495238095238095,
  'f1@30': 0.2814146531142439},
 {'prauc': 0.3575458321391477,
  'p@10': 0.38142857142857145,
  'f1@10': 0.16545285343156016,
  'p@20': 0.3678571428571429,
  'f1@20': 0.24646718097573053,
  'p@30': 0.35000000000000003,
  'f1@30': 0.2890272278222284})

In [11]:
torch.save(model.state_dict(), META / "gnn_best.pt")
